In [4]:
%load_ext autoreload
%autoreload 2
%cd /mnt/sdd1/atharvas/formulacode/datasmith
import datetime
import json
from pathlib import Path

import pandas as pd

curr_date: str = datetime.datetime.now().isoformat()

/mnt/sdd1/atharvas/formulacode/datasmith


In [ ]:
from datasmith.core.models.task import Task
from datasmith.execution.resolution import analyze_commit

commits_df = pd.read_parquet("scratch/artifacts/pipeflush/less_filtered_commits_perfonly.parquet")
tasks = [
    Task(
        owner=row["repo_name"].split("/")[0],
        repo=row["repo_name"].split("/")[1],
        sha=row["sha"],
    )
    for row in commits_df.to_dict(orient="records")
]
print(len(tasks))

19269


In [ ]:
task = tasks[10]
print(task)
task_analysis = analyze_commit(sha=task.sha, repo_name=f"{task.owner}/{task.repo}", bypass_cache=True)
assert task_analysis and task_analysis["can_install"], "Task cannot be installed"
print(task_analysis)

Task(owner='numpy', repo='numpy-financial', sha='8c723d25770fce597175b9d34acffde4560eb521', commit_date=0.0, env_payload='', python_version='', tag='pkg')
{'sha': '8c723d25770fce597175b9d34acffde4560eb521', 'repo_name': 'numpy/numpy-financial', 'package_name': 'numpy-financial', 'package_version': '1.1.0.dev0', 'python_version': '3.12', 'build_command': ['python -m pip install build && python -m build && PIP_NO_BUILD_ISOLATION=false python -m pip wheel --no-deps --no-index -w {build_cache_dir} {build_dir}'], 'install_command': [], 'final_dependencies': ['attrs==23.1.0', 'hypothesis==6.91.1', 'iniconfig==2.0.0', 'numpy==1.26.2', 'packaging==23.2', 'pluggy==1.3.0', 'pytest==7.4.3', 'setuptools==69.0.2', 'sortedcontainers==2.4.0'], 'can_install': True, 'dry_run_log': '\nUsing Python 3.12.11 environment at: /tmp/tmpvxm95kpv/venv_3_12\nResolved 9 packages in 10ms\nWould download 6 packages\nWould install 9 packages\n + attrs==23.1.0\n + hypothesis==6.91.1\n + iniconfig==2.0.0\n + numpy==1.2

In [3]:
results_pth = Path("scratch/artifacts/processed/")
!ls scratch/artifacts/processed/synthetic_commits_perfonly*.parquet

scratch/artifacts/processed/synthetic_commits_perfonly.parquet
scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-11T23:34:10.034989.parquet
scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-12T00:20:02.940307.parquet
scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-12T03:42:49.965377.parquet
scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-12T10:16:06.549234.parquet
scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-12T20:58:19.269085.parquet
scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-12T21:18:35.452250.parquet
scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-12T21:32:05.644755.parquet
scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-13T12:24:05.680135.parquet
scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-13T19:04:14.475606.parquet
scratch/artifacts/processed/

In [3]:
latest_file = sorted(results_pth.glob("synthetic_commits_perfonly*2.parquet"), key=lambda x: x.stat().st_mtime)[-1]
useful_enriched = pd.read_parquet(latest_file)
useful_enriched.head()

,container_name,sha,repo_name,date,kind,has_asv
0,pandas-dev-pandas-11e3dc2738befa5803aea676f1b6...,11e3dc2738befa5803aea676f1b68082a8b24a8b,pandas-dev/pandas,2025-02-10T10:25:51-08:00,commit,True
1,pandas-dev-pandas-1bcdfd05ff4bcb071639c1490998...,1bcdfd05ff4bcb071639c149099876e9cf072aa3,pandas-dev/pandas,2025-04-07T14:21:32-07:00,commit,True
2,pandas-dev-pandas-2096b2897b3692c3f4715e5e65e3...,2096b2897b3692c3f4715e5e65e35fbb01e55eea,pandas-dev/pandas,2025-06-08T08:47:50-04:00,commit,True
3,pandas-dev-pandas-25684be44c950116c5006371c0ab...,25684be44c950116c5006371c0ab5a97c1793108,pandas-dev/pandas,2025-06-16T14:16:29-07:00,commit,True
4,pandas-dev-pandas-4a3fb4bed5bf5315dac37416c42c...,4a3fb4bed5bf5315dac37416c42c9b8c977a3d8c,pandas-dev/pandas,2025-06-05T23:25:18+04:00,commit,True


In [4]:
# convert string to datetime
useful_enriched["mm-yy"] = pd.to_datetime(useful_enriched["date"], errors="coerce", utc=True).dt.strftime("%m-%y")
print(useful_enriched.shape)
useful_enriched

(355, 7)


,container_name,sha,repo_name,date,kind,has_asv,mm-yy
0,pandas-dev-pandas-11e3dc2738befa5803aea676f1b6...,11e3dc2738befa5803aea676f1b68082a8b24a8b,pandas-dev/pandas,2025-02-10T10:25:51-08:00,commit,True,02-25
1,pandas-dev-pandas-1bcdfd05ff4bcb071639c1490998...,1bcdfd05ff4bcb071639c149099876e9cf072aa3,pandas-dev/pandas,2025-04-07T14:21:32-07:00,commit,True,04-25
2,pandas-dev-pandas-2096b2897b3692c3f4715e5e65e3...,2096b2897b3692c3f4715e5e65e35fbb01e55eea,pandas-dev/pandas,2025-06-08T08:47:50-04:00,commit,True,06-25
3,pandas-dev-pandas-25684be44c950116c5006371c0ab...,25684be44c950116c5006371c0ab5a97c1793108,pandas-dev/pandas,2025-06-16T14:16:29-07:00,commit,True,06-25
4,pandas-dev-pandas-4a3fb4bed5bf5315dac37416c42c...,4a3fb4bed5bf5315dac37416c42c9b8c977a3d8c,pandas-dev/pandas,2025-06-05T23:25:18+04:00,commit,True,06-25
...,...,...,...,...,...,...,...
350,joblib-joblib-0957e3125da540ba2715a6fb6fba889a...,0957e3125da540ba2715a6fb6fba889a88ab8720,joblib/joblib,2019-06-18T22:54:36+02:00,commit,True,06-19
351,joblib-joblib-0f1f647a8e2310a2291ea9ffab8c8336...,0f1f647a8e2310a2291ea9ffab8c8336fc01f2c7,joblib/joblib,2019-05-29T17:23:34+02:00,commit,True,05-19
352,joblib-joblib-3169ca6f98fea6dc23a036e74625254b...,3169ca6f98fea6dc23a036e74625254baf5f4ceb,joblib/joblib,2019-05-20T15:24:34+02:00,commit,True,05-19
353,joblib-joblib-79f3fb34721e27e19dc016515c548c66...,79f3fb34721e27e19dc016515c548c66c0acd63c,joblib/joblib,2019-09-10T12:20:03+02:00,commit,True,09-19


In [49]:
useful_enriched_grouped = useful_enriched.groupby(["repo_name", "mm-yy"])["sha"].apply(list).reset_index()
len(useful_enriched_grouped)

147

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from datasmith.execution.temporal_constraints import apply_over_group_batch


def _row_to_dict(row):
    return apply_over_group_batch(row)


results_map = {}
with ThreadPoolExecutor(max_workers=32) as executor:
    for _, row in useful_enriched_grouped.iterrows():
        results_map[row["repo_name"] + "_" + row["mm-yy"]] = executor.submit(_row_to_dict, row)


all_results = {}
for key, future in results_map.items():
    all_results[key] = future.result()

18:51:20 WARNING  simple_useragent.core: Falling back to historic user agent.
18:51:22 ERROR    root: Couldn't load asv.plugins._mamba_helpers because
No module named 'libmambapy'


In [ ]:
from datasmith.agents.utils import _TEST_SUITE_IMPORT_OVERRIDES

all_install_rows = []


def replace_in_list(to_install, constraints, new):
    new_to_install = []
    new_constraints = []
    pkg_name = new.split("==")[0]
    to_install_added = False
    constraints_added = False
    for item in to_install:
        if pkg_name in item:
            new_to_install.append(new)
            to_install_added = True
        else:
            new_to_install.append(item)
    for constraint in constraints:
        if pkg_name in constraint:
            new_constraints.append(new)
            constraints_added = True
        else:
            new_constraints.append(constraint)
    if not to_install_added:
        new_to_install.append(new)
    if not constraints_added:
        new_constraints.append(new)
    return new_to_install, new_constraints


def remove_from_list(lst, pkg_name):
    new_list = []
    for item in lst:
        if "python_version" in item:
            continue
        if item.startswith(pkg_name):
            continue
        new_list.append(item)
    return new_list


for repo_name_mm_yy, results in all_results.items():
    repo_name, mm_yy = repo_name_mm_yy.split("_")
    import_name = Path(repo_name.strip("/ ")).name
    import_name = _TEST_SUITE_IMPORT_OVERRIDES.get(import_name, import_name)
    for r in results:
        majmin = r["published_major_minor"]
        sha = r["sha"]
        to_install = r["to_install"]
        constraints = []
        banned = []
        to_install = [k.replace(">=", "==") for k in to_install]
        constraints = r["constraints"]

        constraints = remove_from_list(constraints, import_name)
        constraints = remove_from_list(constraints, "python_version")
        to_install = remove_from_list(to_install, import_name)
        to_install = remove_from_list(to_install, "python_version")

        if "astropy" in repo_name:
            to_install, constraints = replace_in_list(to_install, constraints, "matplotlib==3.9")
            to_install, constraints = replace_in_list(to_install, constraints, "pytest-astropy")
            to_install, constraints = replace_in_list(to_install, constraints, "numpy==1.26.3")
            # constraints = remove_from_list(constraints, "matplotlib")
            # constraints = remove_from_list(constraints, "pytest-astropy")
            # constraints = remove_from_list(constraints, "numpy")

        if "pandas" in repo_name:
            to_install, constraints = replace_in_list(to_install, constraints, "pyarrow==11.0.0")
            banned = ["pytest-qt"]

        if "dask" in repo_name:
            to_install, constraints = replace_in_list(to_install, constraints, "numpy==1.19")
            # constraints = remove_from_list(constraints, "numpy")

        constraints = list(set(constraints))
        to_install = list(set(to_install))
        banned = list(set(banned))
        for b in banned:
            to_install = remove_from_list(to_install, b)
            constraints = remove_from_list(constraints, b)
        if len(constraints) > 0 or len(to_install) > 0:
            env_payload = json.dumps({
                "constraints": constraints,
                "to_install": to_install,
                "banned": banned,
            })
        else:
            env_payload = ""
        all_install_rows.append({
            "python_version": r["python_version"],
            "repo_name": repo_name,
            "sha": sha,
            "env_payload": env_payload,
            "majmin": majmin,
        })

all_install_rows = pd.DataFrame(all_install_rows)

rows_before = useful_enriched.shape[0]
new_df = pd.merge(useful_enriched, all_install_rows, on=["repo_name", "sha"], how="inner")
rows_after = new_df.shape[0]
print(f"Merged {rows_before - rows_after} rows")
new_df.to_parquet(latest_file.with_suffix(".with_installs.parquet"))
print("saved to ", latest_file.with_suffix(".with_installs.parquet"))

Merged -12 rows
saved to  scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-16T03:26:41.179572.with_installs.parquet


In [48]:
rows_after, rows_before

(367, 355)

In [59]:
# scratch/artifacts/processed/synthetic_commits_perfonly_usefulonly_2025-09-16T03:26:41.179572.with_installs.parquet

new_df.query("sha == 'eb52366465f49d986b06f10592ad530c923ffd50'")["env_payload"].iloc[0]

'{"constraints": ["numpy==1.19"], "to_install": ["numpy==1.19"], "banned": []}'